In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.13 The Hubbard Model: Correlation on a Lattice

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.13",
    title="The Hubbard Model: Correlation on a Lattice",
    blurb="Movement IV opens where the band picture breaks. Fermionic "
    "creation operators are built as explicit matrices — anticommutators "
    "certified at exactly zero error — and assembled into the Hubbard "
    "chain, solved exactly: the dimer against its closed form at 10⁻¹⁵, "
    "then eight sites in a 4900-dimensional sector where a band metal "
    "turns Mott insulator (charge gap growing to U − 4t while the spin "
    "gap stays put), double occupancy is squeezed out, and the surviving "
    "spins secretly enact a Heisenberg antiferromagnet with J = 4t²/U — "
    "matched to a separate Heisenberg diagonalization at the percent "
    "level. Anderson's superexchange, computed.",
    difficulty="advanced",
    estimate="150–180 min",
)

## Notebook overview

Movement III's story had one silent assumption: an electron's energy
depends on which *band state* it occupies, not on where the *other*
electrons are. [§8.8](exact-conditions-band-gap.ipynb) already showed the
assumption failing by exactly the derivative discontinuity; this movement
meets the failure head-on. The minimal arena is Hubbard's 1963 model
{cite}`hubbard1963`: take the Wannier orbitals of
[§8.12](berry-wannier-ssh.ipynb) — one localized orbital per site, with a
known center — keep only nearest-neighbor hopping $t$ and the Coulomb
repulsion $U$ *between two electrons on the same orbital*, and discard
everything else. Two parameters. The model contains band theory ($U = 0$),
the atomic limit ($t = 0$), and, in between, the physics band theory
cannot reach: the Mott insulator — a system with a half-filled band, which
[§7.12](../07-quantum-statistical-mechanics/bloch-theorem-band-structure.ipynb)
would declare a metal, that *insulates because the electrons repel*.

The build is exact diagonalization, the many-body method with no
approximations to apologize for. First the operators themselves:
$\hat c_i^\dagger$ as explicit matrices on the Fock space of
[§7.23](../07-quantum-statistical-mechanics/second-quantization.ipynb),
with the Jordan–Wigner sign counted by `popcount` — and every
anticommutator certified at *exactly* zero error, since sign bookkeeping
is integer arithmetic. Then the Hubbard dimer, the model's hydrogen
molecule, against its closed-form ground state at $10^{-15}$ across five
values of $U$, with the superexchange energy $-4t^2/U$ emerging at large
$U$. Then the real thing: the $L = 8$ chain at half filling, a
$\binom{8}{4}^2 = 4900$-dimensional $S_z = 0$ sector diagonalized by
`scipy.sparse.linalg.eigsh` in a tenth of a second, benchmarked against
the $U = 0$ closed form $-(1+\sqrt2)/2$ per site and against Lieb and
Wu's exact Bethe-ansatz solution {cite}`liebwu1968` (our eight sites land
within $0.3\%$ of their infinite chain). The physics follows in three
measured acts: double occupancy squeezed from $1/4$ to $0.01$ as $U$
grows; the charge gap opening to $U - 4t$ while the spin gap stays
bounded — the Mott insulator, gapped for charge and soft for spin; and
Anderson's superexchange {cite}`anderson1959` — the ground-state spin
correlations of the $U = 32$ Hubbard chain match an independently
diagonalized Heisenberg antiferromagnet with $J = 4t^2/U$ at the percent
level, with the deviation falling fourfold per doubling of $U$, exactly as
second-order perturbation theory promises.

> **Conventions (this notebook).** Hopping $t = 1$ sets the energy unit;
> chains are periodic unless stated (the dimer is open). Site occupations
> are bit masks (bit $i$ set $=$ orbital $i$ occupied), one integer per
> spin species, with the orbital ordering fixed once: all spin-up modes
> before all spin-down. Fermionic signs are Jordan–Wigner parities
> computed by `int.bit_count` of the mask below the acted-on bit. Sector
> bases are built with `itertools.combinations`, Hamiltonians as
> `scipy.sparse` matrices, ground states by `eigsh(k=1, which="SA")`
> (dense `numpy.linalg.eigh` where the sector is small).
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: an exact anticommutator, a closed-form
> eigenvalue, a Bethe-ansatz benchmark, a second-order scaling law. A ✓
> is strong evidence; a ✗ is a prompt to locate the discrepancy, not an
> automatic verdict.
>
> **Scope.** The one-band, one-dimensional, nearest-neighbor Hubbard
> model at and near half filling, by exact diagonalization. Doped chains,
> higher dimensions (where the model is believed to hold cuprate
> superconductivity's secrets, and is *not* solved), and the
> quantum Monte Carlo and DMRG methods that reach them are surveyed in
> Martin, Reining, and Ceperley {cite}`martinreining2016` Ch. 3 and
> Vanderbilt-adjacent reviews; the
> spectral-function view of this same model is
> [§8.14](quasiparticles-gw.ipynb)'s business.

## Theory in brief

### From Wannier orbitals to two parameters

Expand the field operators of
[§7.23](../07-quantum-statistical-mechanics/second-quantization.ipynb) in
a basis of localized Wannier orbitals $w_i$ and the full interacting
Hamiltonian becomes hoppings $t_{ij}$ plus a four-index Coulomb tensor
$U_{ijkl}$. Hubbard's cut {cite}`hubbard1963`: for well-localized
orbitals the on-site element $U_{iiii} \equiv U$ dwarfs all others
(neighboring-orbital repulsion is screened; exchange is smaller still),
and $t_{ij}$ beyond nearest neighbors decays exponentially
([§8.12](berry-wannier-ssh.ipynb) measured that decay). What survives is

```{math}
:label: eq-hub-model
\hat H \;=\; -t \sum_{\langle ij\rangle\sigma}
\left(\hat c^\dagger_{i\sigma}\hat c_{j\sigma} + \text{h.c.}\right)
\;+\; U \sum_i \hat n_{i\uparrow}\hat n_{i\downarrow} ,
```

the minimal Hamiltonian in which kinetic delocalization and local
repulsion compete. At $U = 0$ it is the tight-binding chain of
[§8.9](tight-binding.ipynb), bandwidth $4t$; at $t = 0$ each site holds
$0$, $1$ ($E = 0$), or $2$ ($E = U$) electrons and half filling puts
exactly one electron per site: an insulator with no kinetic energy at
all. The interesting physics is the war between the limits, and the
half-filled chain fights it at every $U$.

### The dimer: the model's hydrogen molecule

Two sites, one electron of each spin, $S_z = 0$: a four-state problem.
In the basis $\{|\!\uparrow,\downarrow\rangle, |\!\downarrow,\uparrow
\rangle, |\!\uparrow\downarrow, 0\rangle, |0, \uparrow\downarrow\rangle\}$
the Hamiltonian is a $4\times4$ matrix whose singlet sector gives the
closed-form ground state

```{math}
:label: eq-hub-dimer
E_0 \;=\; \frac{U - \sqrt{U^2 + 16\,t^2}}{2}
\;\;\xrightarrow{\;U \gg t\;}\;\; -\,\frac{4t^2}{U} :
```

at large $U$ the covalent bond gives way to *superexchange*
{cite}`anderson1959` — the two spins can no longer share a site, but a
virtual double occupation (cost $U$, amplitude $t$, twice) still lowers
the singlet by $4t^2/U$ while leaving the triplet untouched.
Antiferromagnetism without any magnetic interaction in the Hamiltonian.

### The Mott insulator, and the effective spin model

At half filling the charge gap
$\Delta_c = E_0(N{+}1) + E_0(N{-}1) - 2E_0(N)$ — the cost of moving one
electron across the system, [§8.8](exact-conditions-band-gap.ipynb)'s
gap, now for an interacting Hamiltonian — opens for *any* $U > 0$ in one
dimension (Lieb–Wu {cite}`liebwu1968`) and approaches $U - 4t$ at strong
coupling: adding an electron costs the repulsion $U$, minus the $4t$
a doubly-occupied site recovers by delocalizing. The spin sector pays no
such toll: flipping spins moves no charge, so the spin gap stays of
order $J$ and vanishes as $L \to \infty$. One material, two energy
scales. The surviving low-energy physics is spins only: second-order
perturbation theory in $t/U$ (each neighboring pair enacting the dimer's
virtual hop) maps the half-filled model onto the Heisenberg
antiferromagnet

```{math}
:label: eq-hub-heisenberg
\hat H_{\mathrm{eff}} \;=\; J \sum_{\langle ij\rangle}
\left(\hat{\mathbf S}_i \cdot \hat{\mathbf S}_j - \tfrac14\right),
\qquad J = \frac{4t^2}{U},
```

with corrections entering at relative order $(t/U)^2$. Exercise 6 tests
the map where it can be tested: same sizes, both models solved exactly,
correlation functions side by side.

## Setup

Data and instruments: the hopping unit, the Jordan–Wigner parity of
[§7.23](../07-quantum-statistical-mechanics/second-quantization.ipynb)
restated in bit arithmetic, a basis enumerator, and a ground-state solver
wrapper. The notebook's own machinery — the fermionic operators and the
sector Hamiltonian itself — you build in Exercises 1 and 2.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

T_HOP = 1.0  # data: the hopping integral — the energy unit throughout


# built from scratch in §7.23 (jw_ops, the string of Z's, unit-tested by the
# anticommutator table); restated here in bit arithmetic as an instrument.
def jw_sign(mask, i):
    """Jordan-Wigner sign (-1)^(number of occupied modes below bit i).

    The fermionic sign an operator acting on mode i picks up from
    anticommuting past every occupied mode of lower index.

    Parameters
    ----------
    mask : int
        Occupation bit mask.
    i : int
        Mode index being acted on.

    Returns
    -------
    float
        +1.0 or -1.0.
    """
    return -1.0 if (mask & ((1 << i) - 1)).bit_count() & 1 else 1.0


# instrument: combinatorial plumbing — enumerating fixed-occupation bit
# masks is bookkeeping, not the lesson of any exercise here.
def basis_masks(n_modes, n_occ):
    """All bit masks of n_modes bits with exactly n_occ bits set.

    Parameters
    ----------
    n_modes : int
        Number of orbitals (bits).
    n_occ : int
        Number of occupied orbitals.

    Returns
    -------
    list of int
        The masks, in the canonical combinations order.
    """
    return [sum(1 << i for i in combo) for combo in combinations(range(n_modes), n_occ)]


# instrument: a solver dispatch wrapper (dense below dimension 200, sparse
# Lanczos with a deterministic start vector above) — plumbing, not the
# lesson. It diagonalizes whatever `hubbard_sector` is bound to when it
# runs: the one you build in Exercise 2 (Python looks the name up at call
# time, not at definition time).
def ground_state(n_sites, n_up, n_dn, u_int, pbc=True):
    """Ground energy and vector of a Hubbard sector.

    Uses sparse Lanczos (eigsh, which="SA") for sectors above dimension
    200 and dense numpy.linalg.eigh below.

    Parameters
    ----------
    n_sites, n_up, n_dn : int
        Chain length and electron numbers.
    u_int : float
        On-site repulsion.
    pbc : bool, optional
        Boundary condition.

    Returns
    -------
    tuple
        (E0, ground vector, up masks, down masks).
    """
    ham, ups, dns = hubbard_sector(n_sites, n_up, n_dn, u_int, pbc)
    if ham.shape[0] <= 200:
        vals, vecs = np.linalg.eigh(ham.toarray())
        return float(vals[0]), vecs[:, 0], ups, dns
    # fixed start vector: the U = 0 sector has an open-shell ground-state
    # degeneracy (the k = +-pi/2 levels sit at zero energy), and a
    # symmetric, deterministic v0 selects the same symmetric member on
    # every run instead of whatever Lanczos's random start lands on
    vals, vecs = eigsh(ham, k=1, which="SA", v0=np.ones(ham.shape[0]))
    return float(vals[0]), vecs[:, 0], ups, dns

## Exercise 1 — The operators, certified at zero

Everything downstream rests on getting fermionic signs right, so the
first exercise builds $\hat c_i$ as an explicit matrix and checks the
algebra — not to a tolerance, but *exactly*, because Jordan–Wigner signs
are integer arithmetic.

**Part a)** On the $2^3$-dimensional Fock space of three spinless modes,
build the annihilation matrices: $\hat c_i$ maps the basis state with
mask $s$ (bit $i$ set) to the state with mask $s \oplus 2^i$, weighted by
the Jordan–Wigner parity `jw_sign(s, i)`
([§7.23](../07-quantum-statistical-mechanics/second-quantization.ipynb)'s
convention, now executable).

**Part b)** Verify the full algebra with `numpy` matrix products: all
nine $\{\hat c_i, \hat c^\dagger_j\} = \delta_{ij}\mathbb{1}$ and all
nine $\{\hat c_i, \hat c_j\} = 0$, with the *maximum absolute deviation
equal to* $0.0$ — floating point can represent these signs exactly, so
any nonzero deviation means a sign error, full stop. Confirm also that
dropping the string (using $+1$ in place of the parity) *breaks* the
$i \neq j$ anticommutators: the string is load-bearing.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the algebra is exact, and the string is necessary

Zero means zero here: both anticommutator families at machine-exact
$0.0$, and the string-free counterfeit failing at $O(1)$.

In [ ]:
validate.check(
    bool(dev_delta == 0.0 and dev_zero == 0.0),
    "all 18 anticommutators exact to the last bit",
    f"deviations {dev_delta}, {dev_zero}",
)
validate.check(
    bool(broken >= 1.0),
    "dropping the Jordan-Wigner string breaks the algebra",
    f"max violation {broken}",
)

## Exercise 2 — The Hubbard dimer, exactly

The model's hydrogen molecule, solved both ways — by machinery you must
first build.

**Part a)** Write `hubbard_sector(n_sites, n_up, n_dn, u_int, pbc=True)`,
the sparse Hubbard Hamiltonian {eq}`eq-hub-model` in the fixed
$(N_\uparrow, N_\downarrow)$ sector: basis states as the product of an
up-spin mask and a down-spin mask (flattened as
$a \cdot D_\downarrow + b$), the interaction diagonal ($U$ per doubly
occupied site), and hopping within each spin species carrying the
Jordan–Wigner parity of the bits strictly *between* the two sites — the
boundary bond of a periodic chain crosses the string, so its sign counts
every mode in between, which is precisely the subtlety Exercise 1
certified in miniature. **Write this one yourself** — the implementation
is the lesson. Its acceptance test is the dimer: solve the two-site,
one-up-one-down system (open boundary) via the Setup's `ground_state`
(which now drives *your* machinery) and compare with
Eq. {eq}`eq-hub-dimer` at
$U = 0, 1, 4, 8, 20$: agreement at $10^{-15}$, five for five.

**Part b)** Extract the double occupancy
$d = \langle \hat n_{i\uparrow}\hat n_{i\downarrow}\rangle$ per site from
the ground vector. At $U = 0$ the two electrons ignore each other and
$d = 1/4$ exactly (each site's up and down occupations independent, each
$1/2$); repulsion strangles it.

**Part c)** Watch superexchange emerge: at $U = 10, 20, 40$ compare
$E_0$ with $-4t^2/U$ — the ratio must walk monotonically to $1$
($0.963$, $0.990$, $0.9975$: the second-order correction $4t^2/U^2$ dying as promised),
the singlet's virtual-hopping reward.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — the dimer knows its closed form

Five exact energies at $10^{-14}$, the free-electron $d = 1/4$, and the
superexchange ratio marching monotonically to $1$.

In [ ]:
validate.check(
    bool(dimer_dev < 1e-13),
    "dimer ED = closed form at U = 0, 1, 4, 8, 20",
    f"worst |dE| = {dimer_dev:.1e}",
)
validate.close(docc_free, 0.25, "U = 0 double occupancy = 1/4", rtol=1e-12)
validate.check(
    bool(
        super_ratios[0] < super_ratios[1] < super_ratios[2]
        and abs(super_ratios[2] - 1.0) < 3e-3
    ),
    "E0 -> -4t^2/U monotonically (0.3% by U = 40t)",
    f"ratios {super_ratios[0]:.4f} -> {super_ratios[2]:.4f}",
)

## Exercise 3 — Eight sites: the sector, benchmarked twice

Now the machinery earns its keep: the half-filled $L = 8$ periodic chain.

**Part a)** Count before computing: the $S_z = 0$, $N = 8$ sector is
$\binom{8}{4}^2 = 4900$-dimensional (the full Fock space is $4^8 =
65{,}536$; symmetry buys a factor of $13$). Build the sector Hamiltonian
with the `hubbard_sector` you wrote in Exercise 2 and confirm dimension
and Hermiticity (`scipy.sparse` max asymmetry, exactly $0$).

**Part b)** Benchmark against the two ends of the coupling axis. At
$U = 0$ the periodic tight-binding levels $-2t\cos(2\pi m/8)$ filled
with four electrons per spin give
$E_0/L = -(1 + \sqrt2)/2 = -1.20711$ *exactly* (the $k = \pm\pi/2$
levels sit at zero energy, so the open-shell degeneracy costs nothing);
`eigsh` must land on it at $10^{-9}$. At $U = 4$ compare with the exact
infinite-chain Bethe-ansatz energy of Lieb and Wu {cite}`liebwu1968`,
$-0.573729$ per site: eight sites land at $-0.575441$, within $0.3\%$ —
exact diagonalization touching an exact solution.

In [ ]:
# (solution hidden on the public site)


### Validation 3 — counted, symmetric, and twice benchmarked

Dimension $4900$, symmetry exact, the $U = 0$ closed form at $10^{-9}$,
and Lieb–Wu within half a percent from eight sites.

In [ ]:
validate.check(
    bool(sector_dim == 4900 and asym == 0.0),
    "sector dimension 4900 and H exactly symmetric",
    f"dim {sector_dim}, asym {asym}",
)
validate.close(
    e_free, e_free_exact, "U = 0 energy = -(1+sqrt(2))/2 per site", rtol=1e-9
)
validate.check(
    bool(finite_size < 5e-3),
    "eight sites within 0.5% of the infinite-chain Bethe ansatz (U = 4)",
    f"deviation {100*finite_size:.2f}%",
)

## Exercise 4 — The crossover: kinetic energy pays the repulsion's toll

With the machinery certified, sweep the war between the limits.

**Part a)** For $U = 0$ to $16$, compute $E_0/L$ and the double
occupancy of the half-filled chain. Both must fall monotonically: energy
because $U$ only adds a positive diagonal, double occupancy because that
is *how* the ground state dodges the toll — from exactly $1/4$ at
$U = 0$ (the uncorrelated value; the free chain's open-shell degeneracy
is resolved deterministically by the fixed Lanczos start vector) to
$0.011$ at $U = 16$, a $96\%$ eviction.

**Part b)** The eviction is the Mott localization in slow motion: plot
both curves, and against the $E_0/L$ curve mark the two asymptotes it
interpolates — the free value $-(1+\sqrt2)/2$ and the superexchange
energy per site of the effective Heisenberg model (Exercise 6's
$J(E_{\mathrm{Heis}} - L/4)/L$ with $J = 4t^2/U$).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — both monotone, and quantitatively evicted

Energy and double occupancy strictly falling across the nine couplings;
$d(U{=}16) \approx 0.011$, and the large-$U$ energy within $2\%$ of the
Heisenberg asymptote.

In [ ]:
validate.check(
    bool(np.all(np.diff(energy_per_site) > 0.0) and np.all(np.diff(docc_sweep) < 0.0)),
    "E0/L rises and d falls monotonically with U",
    f"d: {docc_sweep[0]:.3f} -> {docc_sweep[-1]:.3f}",
)
validate.close(float(docc_sweep[-1]), 0.0106, "d(U = 16) (96% eviction)", rtol=5e-2)
validate.close(
    float(energy_per_site[-1]),
    float((4.0 / 16.0) * (E_HEIS_8 - 2.0) / 8.0),
    "E0/L(U = 16) on the superexchange asymptote",
    rtol=2e-2,
)

## Exercise 5 — The Mott gap: charge pays, spin does not

[§7.12](../07-quantum-statistical-mechanics/bloch-theorem-band-structure.ipynb)'s
rule — half-filled band, therefore metal — meets its counterexample.

**Part a)** Compute the charge gap
$\Delta_c = E_0(5,4) + E_0(3,4) - 2E_0(4,4)$ (add an electron, remove
one; particle–hole symmetry makes the two sectors' costs equal) at
$U = 2, 4, 8, 16$: it grows from $0.80$ through $2.01$ and $5.19$ to
$12.66$, closing on the strong-coupling law $U - 4t$ (deviation $0.65t$
at $U = 16$ and shrinking).

**Part b)** Compute the spin gap $\Delta_s = E_0(5,3) - E_0(4,4)$ — the
cost of one spin flip, no charge moved. It stays below $0.35t$ at every
coupling (and vanishes as $L \to \infty$: the Bethe-ansatz spin sector
is gapless). One system, two prices: this hierarchy *is* the Mott
insulator — insulating for charge transport, alive with spin dynamics.
A free-electron system cannot do this: at $U = 0$ both gaps are finite-
size artifacts of the same magnitude.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — the hierarchy, quantified

Charge gap strictly growing and within $0.7t$ of $U - 4t$ at $U = 16$;
spin gap bounded by $0.35t$ throughout; a factor $\geq 20$ between the
two sectors at strong coupling.

In [ ]:
validate.check(
    bool(np.all(np.diff(charge_gap) > 0.0)),
    "charge gap grows monotonically with U",
    f"{charge_gap[0]:.2f} -> {charge_gap[-1]:.2f}",
)
validate.close(float(charge_gap[-1]), 16.0 - 4.0, "charge gap -> U - 4t", atol=0.7)
validate.check(
    bool(np.max(spin_gap) < 0.35),
    "spin gap bounded (gapless sector in disguise)",
    f"max {np.max(spin_gap):.3f} t",
)
validate.check(
    bool(charge_gap[-1] / spin_gap[-1] > 20.0),
    "charge/spin price ratio exceeds 20 at U = 16",
    f"ratio {charge_gap[-1]/spin_gap[-1]:.0f}",
)

## Exercise 6 — Verdict: the hidden Heisenberg model

Equation {eq}`eq-hub-heisenberg` claims that at strong coupling the
Hubbard chain *is* a spin chain. Both sides can be diagonalized exactly
at $L = 8$; nothing needs to be taken on faith.

**Part a)** Build and solve the $L = 8$ periodic Heisenberg
antiferromagnet ($J = 1$) in its own $S_z = 0$ basis
($\binom{8}{4} = 70$ states): ground energy $E_{\mathrm{Heis}} =
-3.651093$ and the ground-state correlations
$\langle\mathbf S_0\cdot\mathbf S_d\rangle$ for $d = 1$–$4$,
sign-alternating (antiferromagnetic) with slowly decaying magnitude —
the correlations that become Bethe's famous critical spin chain at
$L \to \infty$.

**Part b)** Measure the same correlator in the *Hubbard* ground state
($\hat{\mathbf S}_i$ built from the fermionic operators) at
$U = 4, 8, 16, 32$ and set the two spin chains side by side. The match
must improve as $(t/U)^2$: maximum relative deviation (over the four
distances) $39\%$ at $U = 4$ falling to $1.1\%$ at $U = 32$, roughly
fourfold per doubling.
Check the energies too: $E_0^{\mathrm{Hub}}$ against
$J(E_{\mathrm{Heis}} - L/4)$ at $U = 32$, within half a percent.

**Part c)** Print the verdict table — $U$, $E_0/L$, $d$, $\Delta_c$,
$\Delta_s$, and the Heisenberg deviation — and read it as one story:
repulsion evicts double occupancy, the eviction gaps the charge sector,
and what survives below the gap is an antiferromagnet no term of Eq.
{eq}`eq-hub-model` asked for.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — superexchange at the percent level

Heisenberg ground energy against its known value; alternating signs at
every coupling; the worst-of-four deviation falling as second order demands; and the
energy map closing within half a percent at $U = 32$.

In [ ]:
validate.close(e_heis, -3.651093, "L = 8 Heisenberg ground energy", rtol=1e-6)
validate.check(
    bool(
        all(
            np.all(np.sign(hubbard_corrs[u_int]) == np.array([-1.0, 1.0, -1.0, 1.0]))
            for u_int in u_heis
        )
    ),
    "antiferromagnetic sign alternation at every U",
    "signs -,+,-,+ for d = 1..4",
)
validate.check(
    bool(
        heis_devs[-1] < 1.2e-2
        and all(d1 > d2 for d1, d2 in zip(heis_devs, heis_devs[1:]))
    ),
    "Heisenberg deviation falls monotonically to ~1% at U = 32",
    f"{100*heis_devs[0]:.0f}% -> {100*heis_devs[-1]:.1f}%",
)
validate.check(
    bool(3.0 < heis_devs[-2] / heis_devs[-1] < 5.0),
    "deviation shrinks fourfold per doubling of U (second order in t/U)",
    f"ratio {heis_devs[-2]/heis_devs[-1]:.1f}",
)
validate.close(e_hub32, e_mapped, "E0(U = 32) = J(E_Heis - L/4) within 0.5%", rtol=5e-3)

:::{admonition} With your assistant
:class: tip
Away from half filling the Mott insulator becomes one of the strangest
metals known: in one dimension the electron falls apart into separate
spin and charge excitations. A first look is within your reach: have
your assistant compute the ground energy of the $L = 8$, $U = 8$ chain
with $N = 8, 7, 6$ electrons ($S_z = 0$ or $\pm\tfrac12$ sectors) and
form the chemical potentials $\mu^+ = E_0(N{+}1) - E_0(N)$ and
$\mu^- = E_0(N) - E_0(N{-}1)$ at $N = 7$. Then run the check that is
yours alone: at $N = 7$ (doped) the two chemical potentials must differ
by far less than the half-filled charge gap — `numpy.isclose` of
$\mu^+ - \mu^-$ against $0$ with `atol` a tenth of Exercise 5's
$\Delta_c(U{=}8)$ — the doped chain conducts, the half-filled one
cannot. The check is yours.
:::

## Notebook summary

Movement IV opened with the band picture's minimal enemy: two terms, one
competition. The fermionic algebra was built as matrices and certified
at exactly zero error — 18 anticommutators, integer arithmetic, no
tolerance — with the Jordan–Wigner string shown load-bearing by breaking
it. The dimer matched its closed form $(U - \sqrt{U^2 + 16t^2})/2$ at
$10^{-15}$ five times over and handed us superexchange in miniature
(ratio to $-4t^2/U$ walking $0.963 \to 0.9975$ from $U = 10$ to $40$).
Eight periodic sites at half filling — a 4900-dimensional sector,
`eigsh`, a tenth of a second — hit the $U = 0$ closed form
$-(1+\sqrt2)/2$ at $10^{-10}$ and Lieb–Wu's infinite-chain Bethe ansatz
within $0.3\%$. Then the physics: double occupancy evicted from $1/4$
to $0.011$; the charge gap climbing $0.80 \to 12.66$ onto the $U - 4t$
law while the spin gap idled below $0.35t$ — a system insulating for
charge and gapless for spin, which no independent-electron theory of
this volume can produce; and the buried punchline, Anderson's
superexchange, verified by diagonalizing the Heisenberg chain the
Hubbard model is supposed to become and watching four correlation
functions converge onto it as $(t/U)^2$ — worst-distance deviations
$39\%$, $15\%$, $4\%$, $1.1\%$ — with the ground energies agreeing through $J = 4t^2/U$ to
$0.4\%$. An antiferromagnet emerged from a Hamiltonian containing no
spin coupling whatsoever.

## Outlook

- Everything here was energies and equal-time correlators. The
  *dynamics* — what happens when one electron is injected or removed,
  the spectral function that photoemission actually measures — needs the
  Lehmann machinery of [§8.14](quasiparticles-gw.ipynb), where this same
  $N \pm 1$ sector arithmetic becomes $A(k, \omega)$ and the GW
  approximation is tested against exactly these chains.
- The $4900$-dimensional sector was comfortable; $L = 16$ is
  $165$ million. The exponential wall is real, and the methods that
  negotiate it — quantum Monte Carlo
  ([§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb)
  grown up), DMRG, embedding — define modern many-body practice; Martin,
  Reining, and Ceperley {cite}`martinreining2016` surveys the field
  (Ch. 23–25 for the stochastic methods).
- In two dimensions the doped Hubbard model is the leading candidate
  theory of high-temperature superconductivity — unsolved after four
  decades. The one-dimensional physics computed here (spin–charge
  separation, emergent magnetism) is the solved corner of that problem.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()